In [ ]:
import pandas as pd

In [ ]:
df = pd.read_json('./transformation_rules.json')
df.head()

In [ ]:
# add a rule_id column to the dataframe (two word name like R1, R2, etc.)
df['Rule ID'] = ['R' + str(i) for i in range(1, len(df) + 1)]
df.head()

In [ ]:
df[['Rule ID', 'Group', 'Rule Description', 'Example (Original)', 'Example (Transformed)']].to_csv('Rule_Names.csv', index=False)

In [ ]:
SYSTEM_INSTRUCTION = '''
You are a helpful coding assistant with a 10+ years experience in writing Python code. You will be given a coding task and a set of Transformation Rules along with examples. Analyze the task (no need to write solution), you will have to determine as much Transformation Rules are applicable to the task. Your answer will be preise, consie and return only the Ids, no explanations are needed.

## Transformation Rules
{transformation_rules}
'''

QUERY_PROMPT = '''
## Task
Write Python code for the following method:

```python
{problem_statement}
```

## Response Format
You have to return a json object with the following data:
"valid_rule_ids": [list of rule ids that are applicable to the task],
'''

In [ ]:
from google import genai
from google.genai import types
import os

def get_response_and_tokens(apikey, model, system_instruction, query_prompt):
    client = genai.Client(api_key=apikey)
    
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=query_prompt)],
        )
    ]
    
    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction
        ),
    )
    
    response_text = response.text
    token_counts = {
        'prompt_token_count': response.usage_metadata.prompt_token_count,
        'candidates_token_count': response.usage_metadata.candidates_token_count,
        'total_token_count': response.usage_metadata.total_token_count
    }
    
    return response_text, token_counts

# Usage
apikey = os.getenv("GEMINI_API_KEY")
model = "gemini-2.5-flash"
transformation_rules = df.to_dict(orient='records')
formatted_rules = "\n".join([f"{rule['Rule ID']}: {rule['Rule Description']};\nOriginal Code: \n{rule['Example (Original)']};\nTransformed Code: \n{rule['Example (Transformed)']}" for rule in transformation_rules])

formatted_SYS_PROMPT = SYSTEM_INSTRUCTION.format(transformation_rules=formatted_rules)

print("Formatted SYS PROMPT:")
print(formatted_SYS_PROMPT)

print("Formatted Query:")
print(QUERY_PROMPT)

In [ ]:
df2 = pd.read_parquet('../../../datasets/human_eval_164.parquet')
df2 = df2.iloc[24:27]  # Limit to first 10 rows for testing
df2.head()

In [ ]:
## write the output in separate files
import os
output_folder = '../../../output/actual_humaneval'

for idx, row in df2.iterrows():

    answer = row['prompt'] + '\n\n' + row['canonical_solution']
    with open(os.path.join(output_folder, f'actual_{idx}.py'), 'w') as f:
        f.write(answer)

In [ ]:
df2 = pd.read_parquet('../../../datasets/human_eval_164.parquet')
df2 = df2.sample(3, random_state=42).reset_index(drop=True)
df2.head()

In [ ]:
response_texts = list(dict() for _ in range(len(df2)))
token_counts = list(dict() for _ in range(len(df2)))

for idx, row in df2.iterrows():
    problem_statement = row['prompt']
    formatted_query = QUERY_PROMPT.format(problem_statement=problem_statement)
    print(f"Processing Problem {idx + 1}/{len(df2)}")

    response_text, token_count = get_response_and_tokens(apikey, model, formatted_SYS_PROMPT, formatted_query)
    response_texts[idx] = response_text
    token_counts[idx] = token_count

In [ ]:
for tc in token_counts:
    print(tc)

In [ ]:
response_texts

In [ ]:
import json

# Assuming response_text is one of the JSON strings
# For example, if you have multiple responses:
responses = response_texts  # This should be a list of response texts

def extract_rule_ids(json_string):
    # Remove the markdown code block if present
    if json_string.startswith('```json\n') and json_string.endswith('\n```'):
        json_string = json_string[8:-4]
    # Parse the JSON
    data = json.loads(json_string)
    return data.get('valid_rule_ids', [])

# Extract from all responses
all_rule_ids = []
for resp in responses:
    rule_ids = extract_rule_ids(resp)
    print(rule_ids)

```
{
  "valid_rule_ids": [
    "R24",
    "R41",
    "R42",
    "R44",
    "R45",
    "R46"
  ]
}
{
  "valid_rule_ids": [
    "R2",
    "R4",
    "R6",
    "R8",
    "R11",
    "R16",
    "R41",
    "R42",
    "R43",
    "R44",
    "R45",
    "R46"
  ]
}
{
  "valid_rule_ids": [
    "R1",
    "R4",
    "R11",
    "R17",
    "R25",
    "R27",
    "R37",
    "R41",
    "R42",
    "R45",
    "R46"
  ]
}
```

```
{'prompt_token_count': 4321, 'candidates_token_count': 63, 'total_token_count': 8939}
{'prompt_token_count': 4567, 'candidates_token_count': 101, 'total_token_count': 9724}
{'prompt_token_count': 4293, 'candidates_token_count': 96, 'total_token_count': 11997}
```
